<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20261110/%E6%A9%9F%E6%A2%B0%E5%AD%A6%E7%BF%92%E3%81%AE%E6%BC%94%E7%BF%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 機械学習の演習

#実行環境の準備

In [ ]:
%%capture
!pip -q install japanize-matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

##データの入手

Kaggleの次のデータセットを利用する。

**Telco Customer Churn**

https://www.kaggle.com/datasets/blastchar/telco-customer-churn

ファイル名：

```text
WA_Fn-UseC_-Telco-Customer-Churn.csv
```

KaggleからCSVをダウンロードし、Notebookのファイルにアップロード後、以下のセルを実行すること。

#データの読み込み

In [ ]:
content_dir = Path("/content")
expected_name = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

candidates = list(content_dir.rglob(expected_name))
data_path = candidates[0]

print("使用するファイル:", data_path)

df_raw = pd.read_csv(data_path)

print("データ件数・列数:", df_raw.shape)
#display(df_raw.head())

使用するファイル: /content/WA_Fn-UseC_-Telco-Customer-Churn.csv
データ件数・列数: (7043, 21)


In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


#データセットの説明

Telco Customer Churn データは、
ある通信会社の顧客について、

- 顧客属性
- 契約期間
- 利用している通信・付加サービス
- 契約形態
- 支払方法
- 月額料金・累積料金
- 解約したかどうか

を記録した顧客単位のデータである。
Kaggle版では、**7,043人の顧客、21列**から構成される。

データの1行は、
 **1人の顧客について、その顧客の契約・サービス・料金等の情報と解約結果を記録したもの**
である。

目的変数 `Churn` を

- `No` → 0：解約していない
- `Yes` → 1：解約した

として、
**顧客が解約するかどうかを予測する二値分類**
を考える。

##主な変数

| 変数 | 内容 |
|---|---|
| `customerID` | 顧客を一意に識別するID |
| `gender` | 性別 |
| `SeniorCitizen` | 高齢者かどうか（0/1） |
| `Partner` | パートナーがいるか |
| `Dependents` | 扶養家族がいるか |
| `tenure` | 契約継続月数 |
| `PhoneService` | 電話サービスの契約有無 |
| `MultipleLines` | 複数電話回線の利用 |
| `InternetService` | インターネットサービス種別 |
| `OnlineSecurity` | オンラインセキュリティサービス |
| `OnlineBackup` | オンラインバックアップ |
| `DeviceProtection` | 端末保護サービス |
| `TechSupport` | 技術サポート |
| `StreamingTV` | TVストリーミング |
| `StreamingMovies` | 映画ストリーミング |
| `Contract` | 契約期間（月単位・1年・2年） |
| `PaperlessBilling` | ペーパーレス請求の有無 |
| `PaymentMethod` | 支払方法 |
| `MonthlyCharges` | 月額料金 |
| `TotalCharges` | 累積料金 |
| `Churn` | 解約したか（Yes/No） |

###注意点

`customerID` は顧客を識別するための値であり、説明変数からは除外する。
また、`TotalCharges`
空白を数値変換すると欠損値となる。
この演習では、**欠損値を中央値で補完**
して用いることにする（該当レコードを削除しない）。

In [ ]:
blank_totalcharges = (df_raw["TotalCharges"].astype(str).str.strip() == "").sum()
print("TotalCharges の空白件数:", blank_totalcharges)

TotalCharges の空白件数: 11


#分析テーマ

解約者を予測するモデルを構築し、その精度を評価せよ。
ただし、予測モデルは全体の8割のデータで作成し、
残りの2割のデータで精度を評価するものとする。